In [ ]:
#ruta noche#

ruta = 'C:/Users/controloperativoenel/OneDrive - SEGURIDAD ATLAS LTDA\Escritorio/Stiven/Automatizacion informe Dron/AtlasDescargas/turno_nocturno.csv'

In [12]:
#ruta dia#
ruta = 'C:/Users/controloperativoenel/OneDrive - SEGURIDAD ATLAS LTDA\Escritorio/Stiven/Automatizacion informe Dron/AtlasDescargas/turno_diurno.csv'


<>:2: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:2: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
C:\Users\controloperativoenel\AppData\Local\Temp\ipykernel_3920\2890230238.py:2: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
  ruta = 'C:/Users/controloperativoenel/OneDrive - SEGURIDAD ATLAS LTDA\Escritorio/Stiven/Automatizacion informe Dron/AtlasDescargas/turno_diurno.csv'


In [13]:
frame = pd.read_csv(ruta, encoding='cp1252', sep=';')


In [14]:
import pandas as pd

In [1]:
import pandas as pd
import os

# =========================
# CARPETA DONDE SE DESCARGAN LOS ARCHIVOS
# =========================

carpeta = r'C:/Users/controloperativoenel/OneDrive - SEGURIDAD ATLAS LTDA/Escritorio/Stiven/Automatizacion informe Dron/AtlasDescargas'

# =========================
# BUSCAR ARCHIVO MÁS RECIENTE
# =========================

archivos = [f for f in os.listdir(carpeta) if f.endswith('.csv')]

ruta_completa = max(
    [os.path.join(carpeta, f) for f in archivos],
    key=os.path.getmtime
)

print(f"📂 Archivo detectado: {ruta_completa}")

# =========================
# DETECTAR JORNADA
# =========================

nombre_archivo = os.path.basename(ruta_completa).lower()

if "diurno" in nombre_archivo:
    tipo_jornada = "DIURNO"
elif "nocturno" in nombre_archivo:
    tipo_jornada = "NOCTURNO"
else:
    tipo_jornada = "DESCONOCIDO"

print(f"🕒 Jornada detectada: {tipo_jornada}")

# =========================
# LECTURA SEGURA CSV
# =========================

def leer_csv_seguro(ruta):
    for encoding in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(ruta, encoding=encoding)
        except:
            continue
    return pd.read_csv(ruta, encoding='latin-1', sep=';')

frame = leer_csv_seguro(ruta_completa)

# =========================
# LIMPIEZA COLUMNAS
# =========================

frame = frame.rename(columns={
    "Usuario": "USUARIO PILOTO UAS",
    "SELECCIONE PARQUE FOTOVOLTAICO": "PARQUE SOLAR",
    "JORNADA DE VUELO": "JORNADA",
    "CONDICION DEL CLIMA ES APTO?": "CLIMA_APTO"
})

# =========================
# VUELOS REALIZADOS
# =========================

frame["VUELOS REALIZADOS"] = frame["CLIMA_APTO"].apply(
    lambda x: 1 if str(x).strip().upper() == "SI" else 0
)

# =========================
# AGRUPACIÓN
# =========================

def combinar_parques(parques):
    vistos = []
    for p in parques:
        if p not in vistos:
            vistos.append(p)
    return " / ".join(vistos)

tabla_resumen = (
    frame.groupby(["USUARIO PILOTO UAS"], as_index=False)
    .agg({
        "PARQUE SOLAR": combinar_parques,
        "VUELOS REALIZADOS": "sum"
    })
)

tabla_resumen["TOTAL REGISTROS"] = frame.groupby(
    ["USUARIO PILOTO UAS"]
)["PARQUE SOLAR"].count().values

# =========================
# MINIMOS DINÁMICOS
# =========================

min_vuelos = 8

if tipo_jornada == "DIURNO":
    min_registros = [9, 9, 8, 8, 8, 9, 9, 9, 9]
elif tipo_jornada == "NOCTURNO":
    min_registros = [9, 9, 8, 8, 8, 9, 9, 15, 15]
else:
    raise ValueError("❌ No se pudo identificar si es diurno o nocturno")

tabla_resumen["VUELOS PROGRAMADOS"] = min_vuelos
tabla_resumen["REGISTROS PROGRAMADOS"] = min_registros[:len(tabla_resumen)]

# =========================
# CALCULOS
# =========================

tabla_resumen["EFICACIA VUELOS (%)"] = (
    tabla_resumen["VUELOS REALIZADOS"] / tabla_resumen["VUELOS PROGRAMADOS"]
) * 100

tabla_resumen["CUMPLIMIENTO REGISTROS (%)"] = (
    tabla_resumen["TOTAL REGISTROS"] / tabla_resumen["REGISTROS PROGRAMADOS"]
) * 100

# =========================
# ORDEN
# =========================

orden_usuarios = [
    "Guayepo 1 Enel (Guayepo 1)",
    "Guayepo 12 Enel (Guayepo 12)",
    "GUAYEPO III OPERADOR DRONE 1 (GUAYEPO III OPERADOR DRONE 1)",
    "GUAYEPO III SUPERVISOR DRON (SUPERVISOR GUAYEPO III DRON)",
    "Guayepo Sub Estación EL Uvero Enel (Subestacion El Uvero)",
    "MOTORIZADO 01 FUNDACION -ENEL (MOTORIZADO 1 FUNDACION)",
    "motorizado 2 fundacion (motorizado 2)",
    "Porteria El Paso Enel (Porteriaelpaso)",
    "Porteria PPal La Loma Enel (ppallaloma)"
]

tabla_resumen["USUARIO PILOTO UAS"] = pd.Categorical(
    tabla_resumen["USUARIO PILOTO UAS"],
    categories=orden_usuarios,
    ordered=True
)

tabla_resumen = tabla_resumen.sort_values("USUARIO PILOTO UAS")

# =========================
# TOTAL
# =========================

totales = pd.DataFrame({
    "USUARIO PILOTO UAS": ["TOTAL"],
    "PARQUE SOLAR": [""],
    "VUELOS REALIZADOS": [tabla_resumen["VUELOS REALIZADOS"].sum()],
    "TOTAL REGISTROS": [tabla_resumen["TOTAL REGISTROS"].sum()],
    "VUELOS PROGRAMADOS": [tabla_resumen["VUELOS PROGRAMADOS"].sum()],
    "REGISTROS PROGRAMADOS": [tabla_resumen["REGISTROS PROGRAMADOS"].sum()],
})

totales["EFICACIA VUELOS (%)"] = (
    totales["VUELOS REALIZADOS"] / totales["VUELOS PROGRAMADOS"]
) * 100

totales["CUMPLIMIENTO REGISTROS (%)"] = (
    totales["TOTAL REGISTROS"] / totales["REGISTROS PROGRAMADOS"]
) * 100

tabla_resumen = pd.concat([tabla_resumen, totales], ignore_index=True)

# =========================
# EXPORTAR CON NOMBRE DINÁMICO
# =========================

salida = f"resultado_{tipo_jornada.lower()}.xlsx"
tabla_resumen.to_excel(salida, index=False)

print(f"✅ Archivo generado: {salida}")

📂 Archivo detectado: C:/Users/controloperativoenel/OneDrive - SEGURIDAD ATLAS LTDA/Escritorio/Stiven/Automatizacion informe Dron/AtlasDescargas\turno_diurno.csv
🕒 Jornada detectada: DIURNO
✅ Archivo generado: resultado_diurno.xlsx


In [ ]:
import pandas as pd
import numpy